# 소볼 지수 실습

**Sobol Index · Sobol Indices · 분산 기반 전역 민감도**

출력 분산을 입력 변수와 변수 조합의 기여로 분해해 얻는 전역 민감도 지표. 1차 지수는 변수 단독 기여, 총 지수는 상호작용까지 포함한 기여를 나타낸다.

소재 분야에서 이해하기: 공정 변수 5개 중 총 지수가 큰 2개만 정밀 실험 대상으로 남긴다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [SALib 민감도 분석 문서](https://salib.readthedocs.io/en/latest/)

## 1. 소볼 지수를 직접 계산

SALib 없이 Saltelli 방식의 1차 지수와 총 지수를 numpy로 계산합니다.
시험 함수는 상호작용이 있는 공정 모형입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def model(x):
    """x1 온도, x2 시간, x3 첨가비율, x4 무관 변수 (모두 0-1)."""
    x1, x2, x3 = x[:, 0], x[:, 1], x[:, 2]
    return 3 * x1 + 0.5 * x2 + 4 * x1 * x3 + 0.2 * x3

names = ['temperature', 'time', 'additive', 'irrelevant']
dimension = 4
n = 8192

A = rng.random((n, dimension))
B = rng.random((n, dimension))
yA, yB = model(A), model(B)
print('출력 분산 %.4f' % yA.var())

In [ ]:
first_order, total_order = [], []
for index in range(dimension):
    AB = A.copy(); AB[:, index] = B[:, index]          # i번째만 B로 교체
    yAB = model(AB)
    variance = yA.var()
    # Saltelli 추정자
    s1 = np.mean(yB * (yAB - yA)) / variance
    st = np.mean((yA - yAB) ** 2) / (2 * variance)
    first_order.append(s1); total_order.append(st)
    print('%-12s 1차 지수 %6.3f   총 지수 %6.3f' % (names[index], s1, st))

print('\n1차 지수 합 %.3f (1보다 작으면 상호작용이 있다는 뜻)' % sum(first_order))
print('총 지수 합 %.3f (1보다 크면 상호작용이 있다는 뜻)' % sum(total_order))

In [ ]:
position = np.arange(dimension)
plt.bar(position - 0.2, first_order, width=0.4, label='first order (S1)')
plt.bar(position + 0.2, total_order, width=0.4, label='total order (ST)')
plt.xticks(position, names, rotation=15); plt.ylabel('Sobol index'); plt.legend(); plt.show()
print('총 지수와 1차 지수의 차이가 큰 변수는 다른 변수와 상호작용하고 있습니다.')
print('총 지수가 0에 가까운 변수는 안심하고 고정할 수 있습니다.')

## 2. 표본 수에 따른 수렴 확인

지수 값은 추정치입니다. 표본이 적으면 흔들립니다.

In [ ]:
for size in (256, 1024, 4096, 16384):
    local = np.random.default_rng(1)
    A2, B2 = local.random((size, dimension)), local.random((size, dimension))
    yA2 = model(A2)
    estimates = []
    for index in range(dimension):
        AB2 = A2.copy(); AB2[:, index] = B2[:, index]
        estimates.append(np.mean((yA2 - model(AB2)) ** 2) / (2 * yA2.var()))
    print('표본 %5d -> 총 지수 %s' % (size, np.round(estimates, 3)))
print('\n실무에서는 SALib 의 sobol.analyze 로 신뢰구간까지 함께 얻습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#sobol-index)을 여세요.